# Proyek: Deteksi Anomali Spektrum RF Cerdas
## 04 - Simulasi Deteksi Anomali Live

**Tujuan Notebook:** Mensimulasikan bagaimana sistem akan beroperasi di dunia nyata. Kita akan memuat model yang telah dilatih sebelumnya untuk menganalisis data 'baru' (file live yang kita sisihkan), mengklasifikasikannya ke dalam profil yang ada, dan memberinya skor anomali.

### Arsitektur Sistem Live

Sistem deteksi live bekerja sebagai berikut:

1.  **Input:** Sebuah file pengukuran spektrum baru.
2.  **Feature Engineering:** Fitur fisis diekstrak dari data baru, sama seperti pada data training.
3.  **Pemuatan Model:** Model `scaler`, `pca`, dan `kmeans` yang sudah terlatih dimuat dari disk.
4.  **Transformasi & Prediksi:** Data baru di-scale dan di-transformasi menggunakan `scaler` dan `pca` yang sudah ada, kemudian `kmeans` memprediksi cluster-nya.
5.  **Pelabelan & Skor Anomali:** Sinyal baru diberi label deskriptif berdasarkan cluster-nya. Skor anomali dihitung sebagai jarak dari sinyal baru ke pusat (centroid) cluster tersebut di ruang fitur. Jarak yang jauh menandakan anomali yang tinggi.

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import joblib
from scipy.signal import find_peaks
from scipy.spatial.distance import euclidean

### Memuat Model-Model Terlatih

In [2]:
MODELS_PATH = '../models/'

try:
    scaler = joblib.load(os.path.join(MODELS_PATH, 'scaler.joblib'))
    pca = joblib.load(os.path.join(MODELS_PATH, 'pca.joblib'))
    kmeans = joblib.load(os.path.join(MODELS_PATH, 'kmeans.joblib'))
    print("Semua model berhasil dimuat.")
except FileNotFoundError:
    print("ERROR: File model tidak ditemukan. Pastikan notebook 03_Model_Training.ipynb sudah dijalankan.")

Semua model berhasil dimuat.


### Menyiapkan Fungsi Analisis & Pelabelan

Kita memerlukan kembali fungsi `extract_physical_features` dan logika pelabelan cerdas dari notebook sebelumnya.

In [3]:
def extract_physical_features(file_path, prominence=5, width=1):
    df = pd.read_csv(file_path, header=None, usecols=[2, 6], names=['frequency', 'power_db'])
    df = df.sort_values(by='frequency').reset_index(drop=True)
    df['power_db'] = pd.to_numeric(df['power_db'], errors='coerce').fillna(-120)
    noise_floor_estimate = df['power_db'].median()
    peaks, properties = find_peaks(df['power_db'], height=noise_floor_estimate + 10, prominence=prominence, width=width)
    extracted_signals = []
    for i, peak_idx in enumerate(peaks):
        freq_mhz = df['frequency'][peak_idx] / 1e6
        power_db = df['power_db'][peak_idx]
        bandwidth_mhz = properties['widths'][i] * 5
        extracted_signals.append({'peak_freq_mhz': round(freq_mhz, 1), 'peak_power_db': round(power_db, 1), 'bandwidth_mhz': round(bandwidth_mhz, 1)})
    return {'file': os.path.basename(file_path), 'noise_floor_db': round(noise_floor_estimate, 1), 'signals': extracted_signals}

def flatten_features(physical_features_list):
    flattened_data = []
    for entry in physical_features_list:
        row = {'file': entry['file'], 'noise_floor_db': entry['noise_floor_db'], 'num_signals': len(entry['signals'])}
        if row['num_signals'] > 0:
            strongest_signal = max(entry['signals'], key=lambda x: x['peak_power_db'])
            row['strongest_signal_power_db'] = strongest_signal['peak_power_db']
            row['strongest_signal_freq_mhz'] = strongest_signal['peak_freq_mhz']
            row['strongest_signal_bw_mhz'] = strongest_signal['bandwidth_mhz']
        else:
            row['strongest_signal_power_db'] = entry['noise_floor_db']
            row['strongest_signal_freq_mhz'] = 0
            row['strongest_signal_bw_mhz'] = 0
        flattened_data.append(row)
    return pd.DataFrame(flattened_data).set_index('file')

In [4]:
def create_smart_labels(kmeans_model, training_features_df):
    interpreted_df = training_features_df.copy()
    interpreted_df['cluster'] = kmeans_model.labels_
    
    freq_allocation_map = {
        (880, 960): "Seluler 2G/3G/4G (Band 8 - GSM)",
        (960, 1400): "Navigasi & Penyiaran (Potensi Interferensi)",
        (1400, 1600): "Satelit & Navigasi (L-Band)",
        (1710, 1880): "Seluler 4G LTE (Band 3)",
    }

    def get_freq_label(freq_mhz):
        for (lower, upper), label in freq_allocation_map.items():
            if lower <= freq_mhz <= upper: return label
        return "Frekuensi Tidak Dikenal"

    cluster_labels = {}
    for i in range(kmeans_model.n_clusters):
        cluster_data = interpreted_df[interpreted_df['cluster'] == i]
        avg_freq = cluster_data['strongest_signal_freq_mhz'].mean()
        avg_power = cluster_data['strongest_signal_power_db'].mean()
        base_label = get_freq_label(avg_freq)
        power_desc = "Sinyal Sangat Kuat" if avg_power > -50 else "Sinyal Kuat" if avg_power > -70 else "Sinyal Normal"
        if avg_freq < 1:
            final_label = f"Profil {i}: Kondisi Baseline (Noise Floor)"
        else:
            final_label = f"Profil {i}: {power_desc} di ~{int(avg_freq)} MHz ({base_label})"
        cluster_labels[i] = final_label
    return cluster_labels

# Buat label dengan memuat data training asli untuk konteks
training_features = pd.read_csv('../data/processed/training_features.csv', index_col='file')
cluster_labels = create_smart_labels(kmeans, training_features)

### Fungsi Analisis Live

In [5]:
def analyze_live_signal(file_path, scaler, pca, kmeans, cluster_labels):
    """Menganalisa satu file sinyal 'live', mengklasifikasikannya, dan menghitung skor anomali."""
    
    # 1. Ekstrak & ratakan fitur fisis
    physical_features = extract_physical_features(file_path)
    live_feature_matrix = flatten_features([physical_features])
    live_numerical_features = live_feature_matrix[training_features.select_dtypes(include=np.number).columns]

    # 2. Gunakan model terlatih untuk transformasi & prediksi
    live_scaled = scaler.transform(live_numerical_features)
    live_pca = pca.transform(live_scaled)
    predicted_cluster = kmeans.predict(live_pca)[0]
    profile_label = cluster_labels[predicted_cluster]
    
    # 3. Hitung Skor Anomali
    centroid_of_predicted_cluster = kmeans.cluster_centers_[predicted_cluster]
    anomaly_score = euclidean(live_pca[0], centroid_of_predicted_cluster)
    
    return {
        "file": os.path.basename(file_path),
        "predicted_profile_label": profile_label,
        "anomaly_score": round(anomaly_score, 2)
    }

### Menjalankan Simulasi

In [6]:
live_files_paths = [
    '../data/raw/1700_anomali_1',
    '../data/raw/1000_anomali_4'
]

print("--- LAPORAN SIMULASI DETEKSI ANOMALI LIVE ---\n")
for live_file in live_files_paths:
    result = analyze_live_signal(live_file, scaler, pca, kmeans, cluster_labels)
    
    print(f"ANALISIS UNTUK FILE: {result['file']}")
    print(f"  -> Profil Terdeteksi : {result['predicted_profile_label']}")
    print(f"  -> Skor Anomali      : {result['anomaly_score']}")
    
    # Memberikan interpretasi skor
    if result['anomaly_score'] > 1.0:
        print("  -> KESIMPULAN        : SANGAT ANOMALI. Sinyal ini sangat berbeda dari semua profil yang dikenal.")
    elif result['anomaly_score'] > 0.5:
        print("  -> KESIMPULAN        : CUKUP ANOMALI. Sinyal ini cocok dengan sebuah profil, namun menunjukkan deviasi yang perlu diperhatikan.")
    else:
        print("  -> KESIMPULAN        : NORMAL. Sinyal ini sesuai dengan profil yang sudah ada.")
    print("-" * 40)

--- LAPORAN SIMULASI DETEKSI ANOMALI LIVE ---

ANALISIS UNTUK FILE: 1700_anomali_1
  -> Profil Terdeteksi : Profil 0: Sinyal Sangat Kuat di ~1639 MHz (Frekuensi Tidak Dikenal)
  -> Skor Anomali      : 0.73
  -> KESIMPULAN        : CUKUP ANOMALI. Sinyal ini cocok dengan sebuah profil, namun menunjukkan deviasi yang perlu diperhatikan.
----------------------------------------
ANALISIS UNTUK FILE: 1000_anomali_4
  -> Profil Terdeteksi : Profil 2: Sinyal Sangat Kuat di ~996 MHz (Navigasi & Penyiaran (Potensi Interferensi))
  -> Skor Anomali      : 0.22
  -> KESIMPULAN        : NORMAL. Sinyal ini sesuai dengan profil yang sudah ada.
----------------------------------------


c:\Users\Achmad Nurnaafi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but KMeans was fitted with feature names
  warnings.warn(
c:\Users\Achmad Nurnaafi\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but KMeans was fitted with feature names
  warnings.warn(
